# Pemodelan Simulasi Sistem Terintegrasi (World Model)
**World Model for Business (Kasus Marketplace Olist)**  

---
Tujuan: Mengintegrasikan model prediksi keterlambatan logistik (Blok 1) dan model prediksi ulasan buruk (Blok 2) ke dalam satu simulator sistem modular yang koheren, memvalidasi integritas arsitekturnya menggunakan uji sintetis terkontrol (*Known-Answer Test*), dan melatih model pada dataset historis Olist untuk persiapan simulasi skenario bisnis.

In [1]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.models.world_model import WorldModelSimulator, run_u2_known_answer_test

sns.set_theme(style='whitegrid', palette='muted')

## 1. Validasi Mesin: Uji Sintetis Jawaban Diketahui (Known-Answer Test)
Sebelum mempercayai simulator multi-tahap untuk simulasi kebijakan bisnis nyata, integritas arsitektur diuji menggunakan data sintetis dengan parameter kausalitas terukur yang diketahui pasti (*ground truth*: efek kausal keterlambatan terhadap ulasan diatur tepat sebesar 40,0%).

In [2]:
print("Menjalankan Validasi Uji Sintetis Jawaban Diketahui...")
ground_truth, mesin_effect, galat = run_u2_known_answer_test()

print(f"\nParameter Asli (Ground Truth) : {ground_truth * 100:.1f}%")
print(f"Estimasi Simulator Mesin      : {mesin_effect * 100:.1f}%")
print(f"Tingkat Galat (Relative Error): {galat * 100:.1f}%")

if galat <= 0.20:
    print("\nUJI SINTETIS VALID: Simulator berhasil merekonstruksi relasi kausalitas laten dengan galat sangat rendah.")
else:
    print("\nUJI SINTETIS GAGAL: Terdeteksi bias estimasi yang signifikan.")

Menjalankan Validasi Uji Sintetis Jawaban Diketahui...

Parameter Asli (Ground Truth) : 40.0%
Estimasi Simulator Mesin      : 40.9%
Tingkat Galat (Relative Error): 2.3%

UJI SINTETIS VALID: Simulator berhasil merekonstruksi relasi kausalitas laten dengan galat sangat rendah.


## 2. Pelatihan & Integrasi Simulator Sistem Bisnis pada Data Riil Olist
Setelah arsitektur simulator terbukti valid, model dilatih secara berantai pada data historis Olist di titik T0 untuk mengintegrasikan variabel penanganan seller, estimasi pengiriman, nilai pesanan, dan ongkir.

In [3]:
df_t0 = pd.read_parquet('../data/processed/features_t0.parquet').dropna(subset=['is_bad_review', 'is_late', 'seller_recent_late_rate_30d', 'estimated_delivery_days'])

X_late = df_t0[['total_items_value', 'total_freight_value', 'estimated_delivery_days', 'seller_recent_late_rate_30d']]
y_late = df_t0['is_late']

X_review = X_late.copy()
X_review['simulated_is_late'] = df_t0['is_late']
y_review = df_t0['is_bad_review']

simulator = WorldModelSimulator()

print("Melatih Blok 1 (Prediksi Keterlambatan Logistik)...")
simulator.train_late_model(X_late, y_late)

print("Melatih Blok 2 (Prediksi Ulasan Buruk)...")
simulator.train_review_model(X_review, y_review)

print("\nSimulator World Model Olist berhasil dilatih dan siap untuk simulasi skenario intervensi bisnis!")

Melatih Blok 1 (Prediksi Keterlambatan Logistik)...


Melatih Blok 2 (Prediksi Ulasan Buruk)...



Simulator World Model Olist berhasil dilatih dan siap untuk simulasi skenario intervensi bisnis!


---
## 3. Ringkasan Arsitektur & Kesiapan Simulasi

1. **Hasil Validasi Arsitektur Simulator**:
   - Parameter Kausalitas Sintetis (*Ground Truth*): **40,0%**
   - Estimasi Rekonstruksi Simulator: **40,9%**
   - Tingkat Galat (*Relative Error*): **2,3%** (sangat jauh di bawah ambang batas toleransi 20%). Mesin terbukti mampu merekonstruksi dinamika kausalitas tanpa halusinasi matematis.

2. **Kesiapan Model Produksi**:
   - Blok 1 (Prediksi Keterlambatan Logistik) dan Blok 2 (Prediksi Ulasan Buruk) telah terintegrasi secara modular.
   - Simulator siap menerima input modifikasi kebijakan bisnis (misalnya penyesuaian SLA janji pengiriman atau intervensi performa penanganan seller) untuk mensimulasikan dampak terhadap kepuasan pelanggan.